# Week 3 — Synthetic Data Generator

LLM-backed **synthetic tabular data** with **Gradio** (preview + CSV download). See [IMPLEMENTATION_PLAN.md](IMPLEMENTATION_PLAN.md) and [TODO.md](TODO.md).

**Open in Colab** (your fork, branch `<brnach_name>`):

`google-colab_link`

Set Colab secrets (or use a local `.env`): `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `HF_TOKEN` (for gated HF weights), optional `GRADIO_USER` / `GRADIO_PASS` for UI login, optional `GRADIO_SHARE` / `SYNTHGEN_GRADIO_SHARE` (`true` / `1` / `yes`) to force a public Gradio tunnel on any runtime.


## Running in Cursor IDE vs Google Colab

**Yes — this notebook can run in Cursor** (and VS Code) using the built-in **Jupyter** view: open the `.ipynb`, pick a **Python kernel** (your venv or system Python), then run cells **top to bottom**.

| Where | What you do |
|--------|----------------|
| **Colab** | Set **Secrets** (`OPENAI_API_KEY`, etc.). By default Gradio uses **`share=True`** (public `*.gradio.live` link). |
| **Cursor / local** | Use a **`.env`** (or shell exports) for the same variable names. By default Gradio uses **`share=False`** (**localhost**; open the printed `http://127.0.0.1:...` URL). |
| **Public link anywhere** | Set **`GRADIO_SHARE=true`** or **`SYNTHGEN_GRADIO_SHARE=true`** in `.env` or Colab secrets to force **`share=True`** on Colab *and* Cursor (creates a `gradio.live` tunnel locally too). |

**Same notebook, no fork:** the `try` / `except ImportError` around `google.colab.userdata` is what makes Colab-only code safe locally.

**Local caveats:** 
`bitsandbytes` + **GPU** paths target **CUDA** (typical on Colab/Linux). 
On **macOS** (Apple Silicon), `hf_local` may install but run on **CPU** (slow) or hit wheel issues — prefer **`gpt` / `claude`** locally unless you have a working PyTorch + optional GPU setup. 
If `%pip` fails in a minimal venv, install deps once with `python -m pip install ...` in that environment, then restart the kernel.


## 1. Install dependencies

Run once per runtime (Colab or local). Installs Gradio, OpenAI SDK, Transformers stack for optional on-GPU HF inference.
This 'pip install' or 'uv add' to be run once in python kernel for installation - not for each run.


In [ ]:
# Install packages into the *active* Jupyter kernel (Colab or Cursor). Re-run after kernel restart if imports fail.
# For a broken %pip, run once in a terminal: python -m pip install "gradio>=4.0" ...
#%pip install -q "gradio>=4.0" "openai>=1.0" python-dotenv pandas huggingface_hub "transformers>=4.40" accelerate bitsandbytes torch


## 2. Environment variables (`.env` or Colab secrets)

Loads keys without printing secret values. Missing keys disable only the backends that need them.


In [ ]:
import os

# HF tokenizers use multi-threaded Rust; forked workers (Gradio/uvicorn) + threads can deadlock — disable tokenizer parallelism early.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Colab: copy "Secrets" into os.environ so downstream code matches local `.env` variable names.
try:
    from google.colab import userdata  # type: ignore

    IN_COLAB = True
    for _secret in (
        "OPENAI_API_KEY",
        "ANTHROPIC_API_KEY",
        "HF_TOKEN",
        "GRADIO_USER",
        "GRADIO_PASS",
        "GRADIO_SHARE",
        "SYNTHGEN_GRADIO_SHARE",
    ):
        try:
            _val = userdata.get(_secret)
            if _val:
                os.environ[_secret] = _val
        except Exception:
            # Missing secret is OK — user may only use hf_local or one paid API.
            pass
except ImportError:
    IN_COLAB = False  # local Jupyter / Cursor: rely on .env or shell exports

from dotenv import load_dotenv

load_dotenv(override=True)  # fills missing keys from `.env` on disk (does not erase Colab secrets already set)


def _mask(k: str) -> None:
    """Debug helper: show which keys exist without leaking full secrets."""
    v = os.getenv(k)
    if v:
        print(f"OK - {k} set (starts {v[:6]}…)")
    else:
        print(f"— {k} not set")


for _k in (
    "OPENAI_API_KEY",
    "ANTHROPIC_API_KEY",
    "HF_TOKEN",
    "GRADIO_USER",
    "GRADIO_PASS",
    "GRADIO_SHARE",
    "SYNTHGEN_GRADIO_SHARE",
):
    _mask(_k)
print("Running in Colab:", IN_COLAB)


## 3. Imports, defaults, and API client registry

Builds OpenAI-compatible clients like `week2_EXERCISE.ipynb` (`MODEL_CONFIG` pattern). Only registers backends whose keys exist.


In [ ]:
import csv
import os
import gc
import io
import json
import re
import tempfile
from typing import Any, Dict, List, Optional, Tuple

import gradio as gr
import pandas as pd
from openai import OpenAI

# Clear any prior Gradio apps in *this* kernel (does not revoke old gradio.live URLs from dead sessions).
gr.close_all()

# --- Tunable limits (safe defaults for Colab Free / classroom use) ---
MAX_ROWS = 500  # upper bound on total synthetic rows per button click
DEFAULT_BATCH = 10  # default objects-per-LLM-call (also capped in UI)
MAX_JSON_RETRIES = 3  # inner-loop retries when JSON parse / keys fail
DEFAULT_HF_MODEL = os.environ.get("SYNTHGEN_HF_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
OPEN_AI_MODEL = os.environ.get("SYNTHGEN_OPENAI_MODEL", "gpt-4o-mini")
ANTHROPIC_MODEL = os.environ.get("SYNTHGEN_ANTHROPIC_MODEL", "claude-sonnet-4-5-20250929")

# Anthropic via OpenAI-compatible HTTP API (same pattern as week2 exercise notebook).
anthropic_url = "https://api.anthropic.com/v1/"
openai_client: Optional[OpenAI] = None
anthropic_openai_compat: Optional[OpenAI] = None

if os.getenv("OPENAI_API_KEY"):
    openai_client = OpenAI()  # default base_url = OpenAI
if os.getenv("ANTHROPIC_API_KEY"):
    anthropic_openai_compat = OpenAI(api_key=os.environ["ANTHROPIC_API_KEY"], base_url=anthropic_url)

# Dropdown label -> (sdk client, model id string) for chat.completions.create
MODEL_CONFIG: Dict[str, Tuple[OpenAI, str]] = {}
if openai_client:
    MODEL_CONFIG["gpt (paid)"] = (openai_client, OPEN_AI_MODEL)
if anthropic_openai_compat:
    MODEL_CONFIG["claude (paid)"] = (anthropic_openai_compat, ANTHROPIC_MODEL)

print("Registered API backends:", list(MODEL_CONFIG.keys()) or "(none — add API keys); hf_local always available if deps load.")


def _env_truthy(name: str) -> bool:
    """Parse common boolean-ish strings from environment variables."""
    v = (os.getenv(name) or "").strip().lower()
    return v in ("1", "true", "yes", "on")

# Gradio public tunnel: on by default in Colab; optional on desktop via secrets / .env (see README).
GRADIO_SHARE = bool(IN_COLAB) or _env_truthy("GRADIO_SHARE") or _env_truthy("SYNTHGEN_GRADIO_SHARE")
print("Gradio share (public link):", GRADIO_SHARE)


### “Batch per LLM”
One LLM call = you send a prompt (plus system message, etc.) and get one completion back.

Batch size (per LLM call) = how many individual records (e.g. rows of synthetic data) you ask that single completion to contain at once.

Example: you want 100 rows. With batch size 10, you might:

Call the model: “Return 10 JSON objects…” → parse → you have 10 rows.
Call again: another 10 → 20 rows total.
Repeat until you reach 100 (or you hit errors / limits).
So “batch per LLM” really means: records per request, not “batch” in the GPU matrix sense (though large prompts still use batching internally on the hardware).

## Why use it at all?
Fewer round trips: batch 20 instead of 1 means fewer API calls for the same total rows (often cheaper and faster than 20× one-row calls).
Bounded output: each answer has a max length (tokens). Asking for 500 rows in one shot often truncates or breaks JSON, so you split into smaller batches.
Easier parsing / retries: if one batch’s JSON is bad, you only re-generate that batch, not the whole dataset.

## Tradeoffs :-

Larger batch per call	:

Fewer calls, often lower cost/latency per row 

Bigger prompt + longer answer → truncation / parse errors more likely

Uses more context window per call


-- 
Smaller batch per call : 

More calls, more overhead

Safer for JSON, easier to retry

Gentler on context limits

-- 

Asking for batch 25 or 50 rows in one generation means a long JSON blob. 

Small models (e.g. 0.5B) struggle to emit long, syntactically perfect JSON end-to-end.
i.e. Smaller batches for hf_local

Use something like 5–10 (not 50) per call for local models. Big batches = long JSON = more truncation and parse errors.

## 4. Field spec, validation, and CSV export

**Field format:** one column per line: `column_name` or `column_name: short description`. Column order in the CSV matches the order in the text box.


In [ ]:
# Regex: one field per line — "columnName" or "columnName: optional description".
_FIELD_LINE = re.compile(r"^\s*([A-Za-z_][A-Za-z0-9_]*)\s*(?::\s*(.*))?\s*$")


def parse_field_spec(text: str) -> Tuple[List[Dict[str, str]], Optional[str]]:
    """Parse multiline field spec into dicts with keys name, desc.

    Returns (fields, error_message). error_message is set on invalid input.
    """
    fields: List[Dict[str, str]] = []
    seen = set()  # CSV headers must be unique
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):  # blank lines or # comments skipped
            continue
        m = _FIELD_LINE.match(line)
        if not m:
            return [], f"Invalid field line (use name or name: desc): {raw_line!r}"
        name, desc = m.group(1), (m.group(2) or "").strip()  # desc optional; feeds LLM prompt later
        if name in seen:
            return [], f"Duplicate column name: {name}"
        seen.add(name)
        fields.append({"name": name, "desc": desc})
    if not fields:
        return [], "Provide at least one field (non-empty lines)."
    return fields, None


def validate_generation_request(fields: List[Dict[str, str]], n_rows: int) -> Optional[str]:
    """Return an error string if inputs are invalid; otherwise None."""
    if n_rows < 1:
        return "Row count must be at least 1."
    if n_rows > MAX_ROWS:  # hard cap defined in config cell — protects spend / runtime
        return f"Row count exceeds cap ({MAX_ROWS}). Lower N or raise MAX_ROWS in the notebook."
    return None  # inputs OK


def rows_to_csv(rows: List[Dict[str, Any]], column_order: List[str]) -> str:
    """Serialize rows to CSV (UTF-8) with stable column order and minimal quoting."""
    buf = io.StringIO()  # build CSV in memory first
    writer = csv.DictWriter(
        buf,
        fieldnames=column_order,
        extrasaction="ignore",  # drop keys not in schema if model hallucinates extras
        lineterminator="\n",
    )
    writer.writeheader()  # first row = user field order
    for r in rows:
        # Align each dict to column_order; .get avoids KeyError on sparse model output
        writer.writerow({k: r.get(k, "") for k in column_order})
    return buf.getvalue()


def write_temp_csv(csv_text: str) -> str:
    """Write CSV text to a temp file and return its path for Gradio download."""
    fd, path = tempfile.mkstemp(suffix=".csv", prefix="synthgen_", text=True)
    with os.fdopen(fd, "w", encoding="utf-8", newline="") as f:
        f.write(csv_text)  # newline="" lets csv module control line endings via lineterminator above
    return path  # pass to gr.File for browser download


## 5. Sanity check (no LLM)

Verifies CSV path without calling any API — catches column-order bugs early.


In [ ]:
# Smoke test CSV path without any LLM: verifies quoting + column order from rows_to_csv / write_temp_csv chain.
_cols = ["id", "label"]
_rows = [{"id": 1, "label": "alpha"}, {"id": 2, "label": "beta,comma"}]  # comma inside cell must survive CSV encoding
_csv = rows_to_csv(_rows, _cols)
assert "label" in _csv and "beta,comma" in _csv
print("CSV sanity OK:", _csv)


## 6. Prompts and JSON extraction

Asks the model for **only** JSON (array of row objects). Strips optional ``` fences; retries are handled in the orchestrator.


In [ ]:
# SYSTEM_SYNTH steers all backends (API + local) toward strict JSON-only synthetic rows.
SYSTEM_SYNTH = """You are a synthetic dataset generator for software testing and education.
Return ONLY valid JSON as specified by the user: no markdown, no code fences, no commentary.
Each generated object must use exactly the requested keys. Values must be plausible but MUST NOT
copy real people names, emails, phone numbers, or addresses (use obviously fake placeholders).
"""


def build_user_prompt(field_defs: List[Dict[str, str]], batch_size: int) -> str:
    """User message: demand exactly `batch_size` objects, keys = schema, root = JSON array."""
    names = [f["name"] for f in field_defs]  # stable key order for CSV + validation
    lines = [f"- {fd['name']}" + (f": {fd['desc']}" if fd["desc"] else "") for fd in field_defs]
    schema = "\n".join(lines)  # human-readable field hints for the model
    return (
        f"Generate exactly {batch_size} diverse JSON objects in a JSON ARRAY (root type array).\n"
        f"Each object MUST contain only these keys, in this order conceptually: {names}.\n"
        f"Field notes:\n{schema}\n"
        f"Output rules: respond with a single JSON array and nothing else."
    )


def strip_code_fences(text: str) -> str:
    """Models often wrap JSON in ``` even when told not — strip one outer fence if present."""
    t = text.strip()
    fence = re.match(r"^```(?:json)?\s*([\s\S]*?)\s*```$", t, re.IGNORECASE)
    if fence:
        return fence.group(1).strip()
    return t


def parse_rows_json(raw: str) -> Tuple[Optional[List[dict]], Optional[str]]:
    """Parse assistant output to Python list[dict]; return (None, err) on structural failure."""
    try:
        data = json.loads(strip_code_fences(raw))
    except json.JSONDecodeError as e:
        return None, f"JSON decode error: {e}"
    if not isinstance(data, list):
        return None, "Expected a JSON array at the top level."
    out: List[dict] = []
    for i, item in enumerate(data):
        if not isinstance(item, dict):
            return None, f"Item {i} is not an object."
        out.append(item)
    return out, None


def validate_row_keys(rows: List[dict], expected: List[str]) -> Tuple[bool, str]:
    """Every row must have exactly the schema keys (extras/forbidden handled here)."""
    exp_set = set(expected)
    for i, row in enumerate(rows):
        keys = set(row.keys())
        if keys != exp_set:
            return False, f"Row {i} keys {sorted(keys)} != expected {expected}"
    return True, ""


## 7. OpenAI / Anthropic chat call

Non-streaming completion for reliable JSON parsing. Uses OpenAI `response_format` JSON object mode with a `rows` wrapper when the backend is OpenAI (required for json_object mode).


In [ ]:
def _openai_messages_array_mode(user_prompt: str) -> list:
    """OpenAI `response_format=json_object` requires a JSON *object* root — wrap table rows under key \"rows\"."""
    wrapped = (
        user_prompt
        + 'Respond as a single JSON object with exactly one key "rows" whose value is the ARRAY of row objects.'
    )
    return [
        {"role": "system", "content": SYSTEM_SYNTH},
        {"role": "user", "content": wrapped},
    ]


def chat_completion_text(client: OpenAI, model: str, user_prompt: str, use_json_object: bool) -> Tuple[Optional[str], Optional[str]]:
    """Single non-streaming chat call; returns text or (None, user-visible error)."""
    if use_json_object:
        messages = _openai_messages_array_mode(user_prompt)
        kwargs = dict(
            model=model,
            messages=messages,
            temperature=0.9,
            response_format={"type": "json_object"},
        )
    else:
        # Non-OpenAI path: ask for raw JSON array in assistant message (no response_format).
        messages = [
            {"role": "system", "content": SYSTEM_SYNTH},
            {"role": "user", "content": user_prompt},
        ]
        kwargs = dict(model=model, messages=messages, temperature=0.9)
    try:
        resp = client.chat.completions.create(**kwargs)
        text = (resp.choices[0].message.content or "").strip()  # choice[0]: single completion
        return text, None
    except Exception as e:
        # HTTP 401/429, DNS, etc. — bubble up as markdown-safe string for Gradio status.
        return None, f"API error: {e!s}"


def unwrap_rows_json(raw: str) -> str:
    """Normalize {\"rows\":[...]} from OpenAI json_object mode into a JSON array string for parse_rows_json."""
    try:
        obj = json.loads(strip_code_fences(raw))
        if isinstance(obj, dict) and "rows" in obj and isinstance(obj["rows"], list):
            return json.dumps(obj["rows"])
    except Exception:
        pass
    return raw  # already an array string or non-JSON — caller may fail parse and retry


## 8. Hugging Face local backend (lazy, optional GPU)

Loads `DEFAULT_HF_MODEL` on first use. Logs in with `HF_TOKEN` if set. Catches OOM with a clear hint.


In [ ]:
import torch
from typing import Any, Optional, Tuple
from huggingface_hub import login as hf_login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

_hf_tok = None  # module-level singletons: load once per kernel
_hf_model = None


def _ensure_hf_login() -> None:
    """HF_TOKEN: gated models + fewer anonymous rate limits on Hub downloads."""
    tok = os.getenv("HF_TOKEN")
    if tok:
        hf_login(token=tok, add_to_git_credential=False)


def ensure_hf_model_loaded() -> Tuple[Optional[Any], Optional[Any], Optional[str]]:
    """Lazy load: expensive download + GPU memory only when user picks hf_local."""
    global _hf_tok, _hf_model
    if _hf_tok is not None and _hf_model is not None:
        return _hf_tok, _hf_model, None
    _ensure_hf_login()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    try:
        _hf_tok = AutoTokenizer.from_pretrained(DEFAULT_HF_MODEL, use_fast=True)
        if device == "cuda":
            quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
            _hf_model = AutoModelForCausalLM.from_pretrained(
                DEFAULT_HF_MODEL,
                quantization_config=quant,
                device_map="auto",  # shards layers across available GPUs (usually one on Colab)
            )
        else:
            _hf_model = AutoModelForCausalLM.from_pretrained(DEFAULT_HF_MODEL, torch_dtype=torch.float32)
            _hf_model.to(device)  # CPU / MPS: slower; OK for tiny models
    except torch.cuda.OutOfMemoryError:
        return None, None, "CUDA OOM while loading HF model — set a smaller SYNTHGEN_HF_MODEL or restart runtime with GPU."
    except Exception as e:
        return None, None, f"HF load error: {e!s}"
    return _hf_tok, _hf_model, None


@torch.inference_mode()  # disable autograd: inference only, less overhead
def hf_generate_text(user_prompt: str) -> Tuple[Optional[str], Optional[str]]:
    """Chat-template prompt -> model.generate -> decode only newly generated tokens."""
    tok, model, err = ensure_hf_model_loaded()
    if err:
        return None, err
    messages = [
        {"role": "system", "content": SYSTEM_SYNTH},
        {"role": "user", "content": user_prompt},
    ]
    try:
        prompt_text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt_text, return_tensors="pt").to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=2048,  # cap completion length — JSON batches need headroom
            do_sample=True,
            temperature=0.9,
            pad_token_id=tok.eos_token_id,
        )
        gen = out[0][inputs["input_ids"].shape[1] :]  # slice off prompt tokens from left
        text = tok.decode(gen, skip_special_tokens=True).strip()
        return text, None
    except torch.cuda.OutOfMemoryError:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return None, "CUDA OOM during generation — reduce batch size or use an API backend."
    except Exception as e:
        return None, f"HF generate error: {e!s}"


## 9. Orchestration — batched generation with retries

Calls the chosen backend until `N` rows are collected. On JSON parse failure, retries with a smaller batch for that step.


In [ ]:
def generate_rows_batched(
    field_defs: List[Dict[str, str]],
    n_rows: int,
    backend: str,
    batch_size: int,
) -> Tuple[List[dict], str]:
    """Accumulate rows until `n_rows` or stop; append human-readable trace into markdown status."""
    names = [f["name"] for f in field_defs]  # canonical column order for CSV + key checks
    rows: List[dict] = []
    status_lines: List[str] = []  # bullet lines shown in Gradio markdown
    remaining = n_rows
    bs = max(1, min(int(batch_size), remaining, 50))  # never ask more than remaining or UI max

    while remaining > 0:
        this_batch = min(bs, remaining)  # rows requested in this outer iteration
        user_prompt = build_user_prompt(field_defs, this_batch)
        raw_out: Optional[str] = None
        last_err: Optional[str] = None
        progressed = False  # did we append any rows this outer iteration?

        for attempt in range(MAX_JSON_RETRIES):
            if backend == "hf_local (free)":
                raw_out, api_err = hf_generate_text(user_prompt)
                use_json_obj = False  # local HF: plain text JSON array in completion
            else:
                if backend not in MODEL_CONFIG:
                    return rows, f"**Error:** backend `{backend}` not configured (missing API key?)."
                client, model_id = MODEL_CONFIG[backend]
                use_json_obj = backend == "gpt (paid)"  # only OpenAI path uses json_object wrapper
                raw_out, api_err = chat_completion_text(client, model_id, user_prompt, use_json_obj)
                if raw_out and use_json_obj:
                    raw_out = unwrap_rows_json(raw_out)  # unwrap {"rows":[...]} -> "[...]" string

            if api_err:
                last_err = api_err
                status_lines.append(f"- attempt {attempt + 1}: {api_err}")
                break  # stop inner retries on hard API failure

            assert raw_out is not None
            parsed, perr = parse_rows_json(raw_out)
            if parsed is None:
                last_err = perr
                if this_batch > 1:
                    this_batch = max(1, this_batch // 2)  # shrink ask — often fixes truncated JSON
                    user_prompt = build_user_prompt(field_defs, this_batch)
                    status_lines.append(f"- JSON parse issue, retrying with batch {this_batch}: {perr}")
                    continue  # same attempt index budget: retry inner for loop
                status_lines.append(f"- giving up on batch: {perr}")
                break

            ok, verr = validate_row_keys(parsed, names)
            if not ok:
                last_err = verr
                status_lines.append(f"- key mismatch: {verr}")
                parsed = None

            if parsed:
                slice_rows = parsed[:this_batch]  # model may return more than asked; clip
                for r in slice_rows:
                    rows.append({k: r.get(k) for k in names})  # normalize key set + column order
                remaining -= len(slice_rows)
                status_lines.append(f"- OK batch size {len(slice_rows)} (remaining {remaining})")
                progressed = True
                last_err = None
                break  # success for this outer while step

        if not progressed and last_err:
            status_lines.append("- stopping: no progress on this batch.")
            break  # outer while: avoid infinite loop when every inner attempt fails

    summary = "## Status\n" + "\n".join(status_lines) if status_lines else "## Status\n(no batches)"
    summary += f"\n\n**Generated rows:** {len(rows)} / {n_rows}"
    if len(rows) < n_rows:
        summary += "\n\n_Partial result — check API keys, model output, or lower batch size._"
    return rows, summary


## 10. Gradio UI — preview + CSV download

Optional HTTP basic auth when `GRADIO_USER` and `GRADIO_PASS` are set. On Colab, `share=True` creates a public gradio.live tunnel.


In [ ]:
# Stop duplicate servers if you re-run this cell in the same kernel before Interrupt.
gr.close_all()


def gradio_generate(field_spec: str, n_rows: float, backend: str, batch_size: float):
    """Bound to Generate: validate -> LLM batches -> markdown status + preview + downloadable CSV."""
    try:
        n_int = int(n_rows)
        bs_int = int(batch_size)
    except (TypeError, ValueError):
        return "**Error:** invalid numeric input.", None, None

    fields, perr = parse_field_spec(field_spec)
    if perr:
        return f"**Error:** {perr}", None, None
    verr = validate_generation_request(fields, n_int)
    if verr:
        return f"**Error:** {verr}", None, None

    if backend == "hf_local (free)":
        _, _, lerr = ensure_hf_model_loaded()  # triggers download on first use
        if lerr:
            return f"**Error:** {lerr}", None, None
    elif backend not in MODEL_CONFIG:
        return f"**Error:** backend `{backend}` unavailable. Add keys or pick hf_local (free) / another backend.", None, None

    rows, md_status = generate_rows_batched(fields, n_int, backend, max(1, bs_int))
    if not rows:
        return md_status, None, None  # still show status markdown even when empty

    order = [f["name"] for f in fields]
    csv_text = rows_to_csv(rows, order)
    path = write_temp_csv(csv_text)
    preview = pd.DataFrame(rows[:20])  # cap preview width for UI
    return md_status, preview, path


_backend_choices = list(MODEL_CONFIG.keys()) + ["hf_local (free)"]
_default_backend = _backend_choices[0] if _backend_choices else "hf_local (free)"

with gr.Blocks(title="Synthetic Data Generator") as demo:
    gr.Markdown("# Synthetic Data Generator\n\nDefine fields, row count, and LLM. CSV preview (first 20 rows) and download.")
    field_box = gr.Textbox(
        label="Fields ( one per line:-    Field Name OR Field Name: description ) - Change the below example as per your need",
        lines=8,
        value="Name: Full name of the employee\n Age: Age in years\n Department: Active department \n assignmentRole: Job title \n Email: Company email address\n Country:ISO Country code\n City:City name as per country code",
    )
    n_box = gr.Number(label="Number of rows", value=5, precision=0, minimum=1, maximum=MAX_ROWS)
    be = gr.Dropdown(choices=_backend_choices, value=_default_backend, label="LLM")
    bs = gr.Number(label="Target batch size (per LLM call) - use smaller batches for hf_local e.g. 5-10", value=DEFAULT_BATCH, precision=0, minimum=1, maximum=50)
    go = gr.Button("Generate")
    status_md = gr.Markdown()
    preview_df = gr.Dataframe(label="Preview (up to 20 rows)", interactive=False)
    out_file = gr.File(label="Download CSV - click at the bottom right")

    go.click(gradio_generate, inputs=[field_box, n_box, be, bs], outputs=[status_md, preview_df, out_file])

_auth = None
if os.getenv("GRADIO_USER") and os.getenv("GRADIO_PASS"):
    _auth = (os.environ["GRADIO_USER"], os.environ["GRADIO_PASS"])  # HTTP basic auth on the Gradio page

# Blocks until Interrupt: share uses Colab default or GRADIO_SHARE env; auth optional.
demo.launch(share=GRADIO_SHARE, auth=_auth)


## 11. Clear out Gradio Live UI , if needed at end of session
Run this to close Gradio UI - when needed at end of sesson


In [ ]:
# Run after Interrupting the launch cell: clears in-process Gradio state (not remote tunnels from old kernels).
#gr.close_all()
